## 1

Загрузите файлы ratings.csv и movies.csv из датасета MovieLens (ml-latest-small). Посчитайте для каждого фильма количество оценок. Отсортируйте фильмы по убыванию количества оценок и выберите топ-10. Объедините с таблицей фильмов, чтобы вывести названия этих десяти фильмов. Выведи результат в понятном формате (например, таблицу с колонками: номер, название фильма, количество оценок, средняя оценка).

In [ ]:
import pandas as pd

ratings = pd.read_csv('ratings.csv')
movies = pd.read_csv('movies.csv')

# Группировка по movieId для подсчета количества и среднего рейтинга
movie_stats = ratings.groupby('movieId')['rating'].agg(['count', 'mean'])

# Сортировка по убыванию количества оценок и выбор топ-10
top_10_ids = movie_stats.sort_values(by='count', ascending=False).head(10)

# Объединение с таблицей movies для получения названий
result = pd.merge(top_10_ids, movies, on='movieId')

# Выбираем нужные колонки и переименовываем их
final_table = result[['title', 'count', 'mean']].copy()
final_table.columns = ['Название фильма', 'Количество оценок', 'Средняя оценка']

# Округляем среднюю оценку до 2 знаков
final_table['Средняя оценка'] = final_table['Средняя оценка'].round(2)

# Добавляем колонку "Номер"
final_table.index = range(1, 11)
final_table.index.name = '№'

print("Топ-10 самых оцениваемых фильмов:")
print(final_table)

## 2

Загрузите файлы ratings.csv и movies.csv из датасета MovieLens (ml-latest-small). Постройте user-item матрицу, где строки — это идентификаторы пользователей, столбцы — идентификаторы фильмов, а значения — рейтинги, которые пользователь поставил фильму. Если пользователь не оценивал фильм, заполните пропуск нулём. Выведите размер полученной матрицы (количество пользователей × количество фильмов) и количество ненулевых элементов.

In [ ]:
import pandas as pd

ratings = pd.read_csv('ratings.csv')
movies = pd.read_csv('movies.csv')

# Построение user-item матрицы
# index — строки (пользователи), columns — столбцы (фильмы), values — значения (рейтинги)
user_item_matrix = ratings.pivot(index='userId', columns='movieId', values='rating').fillna(0)

n_users, n_items = user_item_matrix.shape
nnz_elements = ratings.shape[0]

print(f"Размер полученной матрицы: {n_users} (пользователей) × {n_items} (фильмов)")
print(f"Количество ненулевых элементов (реальных оценок): {nnz_elements}")

print("\nФрагмент матрицы (первые 5 пользователей и 5 фильмов):")
print(user_item_matrix.iloc[:5, :5])

## 3

3. Загрузите файлы ratings.csv и movies.csv из датасета MovieLens (ml-latest-small). Раздели данные ratings.csv на обучающую (80%) и тестовую (20%) выборки случайным образом. На обучающей выборке определите топ-5 самых популярных фильмов (по количеству оценок). Для каждого пользователя из тестовой выборки посчитайте, сколько фильмов из этого топ-5 он оценил (наличие любой оценки в тесте). Precision@5 для пользователя = (число попавших фильмов) / 5. Усредните precision по всем пользователям, у которых есть хотя бы одна оценка в тесте. Выведите среднюю оценку precision@5.

In [ ]:
import io
import zipfile
from urllib.request import urlopen

import numpy as np
import pandas as pd

# --- Скачать ---
url = "https://files.grouplens.org/datasets/movielens/ml-latest-small.zip"
with urlopen(url) as response:
    with zipfile.ZipFile(io.BytesIO(response.read())) as zf:
        with zf.open("ml-latest-small/ratings.csv") as f:
            ratings = pd.read_csv(f)
        with zf.open("ml-latest-small/movies.csv") as f:
            movies = pd.read_csv(f)
# --------------

train, test = np.split(
    ratings.sample(frac=1, random_state=42).reset_index(drop=True),
    [int(len(ratings) * 0.8)],
)

top5_movies = (
    train.groupby("movieId").size().sort_values(ascending=False).head(5).index.tolist()
)

test_users = test.groupby("userId").filter(lambda df: len(df) >= 1)["userId"].unique()
precisions = []
for user_id in test_users:
    user_test_movies = set(test.loc[test["userId"] == user_id, "movieId"])
    hits = len(user_test_movies.intersection(top5_movies))
    precisions.append(hits / 5)

mean_precision_at_5 = np.mean(precisions)
print(f"Средняя precision@5: {mean_precision_at_5:.4f}")

# 4

4. Загрузите файлы ratings.csv и movies.csv из датасета MovieLens (ml-latest-small). Создайте неявную матрицу: если рейтинг пользователя ≥ 4, поставьте 1 (сильный положительный сигнал), если рейтинг < 4, поставьте 0 (слабый сигнал или отсутствие интереса). Постройте матрицу пользователь–фильм на основе этих бинарных значений (пропуски заполните 0). Вычислите плотность матрицы (доля единиц среди всех элементов) и выведите её.

In [ ]:
import io
import zipfile
from urllib.request import urlopen

import numpy as np
import pandas as pd

# --- Скачать ---
url = "https://files.grouplens.org/datasets/movielens/ml-latest-small.zip"
with urlopen(url) as response:
    with zipfile.ZipFile(io.BytesIO(response.read())) as zf:
        with zf.open("ml-latest-small/ratings.csv") as f:
            ratings = pd.read_csv(f)
        with zf.open("ml-latest-small/movies.csv") as f:
            movies = pd.read_csv(f)
# --------------

ratings["implicit"] = (ratings["rating"] >= 4).astype(int)

user_movie = ratings.pivot_table(
    index="userId",
    columns="movieId",
    values="implicit",
    aggfunc="max",
    fill_value=0,
)

density = user_movie.values.sum() / user_movie.size
print(f"Плотность матрицы: {density:.6f}")

# 5🍓

5. С применением библиотеки Surprise загрузите ratings.csv с помощью Reader (шкала оценок от 0.5 до 5.0). Разделите данные на обучающую (80%) и тестовую (20%) выборки. Создайте модель KNNBasic (user-based, используйте косинусное сходство, количество соседей k = 40). Обучите модель на обучающей выборке. Сделайте прогнозы для тестовой выборки и вычислите RMSE (среднеквадратичную ошибку). Выведите значение RMSE.

In [ ]:
# ==========================================
# 0. ОБЯЗАТЕЛЬНО
# ==========================================
!pip uninstall numpy -y
!pip install numpy==1.26.4
!pip install scikit-surprise

In [ ]:
import zipfile
from io import BytesIO
import pandas as pd
import requests
from surprise import KNNBasic, Dataset, Reader, accuracy
from surprise.model_selection import train_test_split

# ==========================================
# 1. Загрузка и подготовка данных
# ==========================================
print("Скачивание и подготовка датасета MovieLens...")
url = "https://files.grouplens.org/datasets/movielens/ml-latest-small.zip"
response = requests.get(url)
zip_file = zipfile.ZipFile(BytesIO(response.content))

# Загрузка ratings.csv и movies.csv в Pandas DataFrame
ratings_df = pd.read_csv(zip_file.open("ml-latest-small/ratings.csv"))
movies_df = pd.read_csv(zip_file.open("ml-latest-small/movies.csv"))

print(f"Загружено оценок: {len(ratings_df)}")
print(f"Загружено фильмов: {len(movies_df)}")

# Настройка Reader с диапазоном оценок от 0.5 до 5.0
reader = Reader(rating_scale=(0.5, 5.0))

# Загрузка данных в Surprise (требуется строгий порядок колонок: пользователь, товар, рейтинг)
dataset = Dataset.load_from_df(
    ratings_df[["userId", "movieId", "rating"]], reader
)

# ==========================================
# 2. Разделение данных на выборки (80/20)
# ==========================================
# Используется train_test_split из surprise.model_selection
trainset, testset = train_test_split(dataset, test_size=0.2, random_state=42)

# ==========================================
# 3. Создание и обучение модели KNNBasic
# ==========================================
# Настройка опций сходства: косинусное сходство (cosine), ориентированное на пользователей (user-based)
sim_options = {"name": "cosine", "user_based": True}

# Инициализация KNNBasic с k=40 соседей
model = KNNBasic(k=40, sim_options=sim_options)

print("Обучение модели KNNBasic...")
model.fit(trainset)

# ==========================================
# 4. Прогнозирование и оценка качества
# ==========================================
print("Тестирование модели...")
predictions = model.test(testset)

# Вычисление и вывод метрики RMSE
rmse = accuracy.rmse(predictions)
print(f"Итоговое значение RMSE на тестовой выборке: {rmse:.4f}")

Скачивание и подготовка датасета MovieLens...
Загружено оценок: 100836
Загружено фильмов: 9742
Обучение модели KNNBasic...
Computing the cosine similarity matrix...
Done computing similarity matrix.
Тестирование модели...
RMSE: 0.9823
Итоговое значение RMSE на тестовой выборке: 0.9823


# 6🍓

6. С применением библиотеки Surprise загрузите встроенный датасет ml-100k. Разделите данные на обучающую (80%) и тестовую (20%) выборки. Создайте модель SVD — это реализация FunkSVD. Установи параметры: n_factors = 50, n_epochs = 20, lr_all = 0.005, reg_all = 0.02. Обучите модель на обучающей выборке. Сделайте прогнозы для тестовой выборки. Вычислите RMSE и MAE. Дополнительно сделайте прогнозы для одного конкретного пользователя и одного фильма (например, userId = 1, movieId = 50) и выведите предсказанный рейтинг.

In [ ]:
# ==========================================
# 0. ОБЯЗАТЕЛЬНО
# ==========================================
!pip uninstall numpy -y
!pip install numpy==1.26.4
!pip install scikit-surprise

Found existing installation: numpy 2.0.2
Uninstalling numpy-2.0.2:
  Successfully uninstalled numpy-2.0.2
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 67.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pytensor 2.38.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
shap 0.52.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
tifffile 2026.4.11 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; py

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 49.9 MB/s eta 0:00:00


In [ ]:
from surprise import Dataset, SVD, accuracy
from surprise.model_selection import train_test_split

# ==========================================
# 1. Загрузка встроенного датасета ml-100k
# ==========================================
print("Загрузка встроенного датасета MovieLens-100k...")
data = Dataset.load_builtin("ml-100k")

# ==========================================
# 2. Разделение данных на выборки (80/20)
# ==========================================
# test_size=0.2 задает 20% тестовой выборки, random_state зафиксирован для воспроизводимости
trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

# ==========================================
# 3. Создание и обучение модели SVD (FunkSVD)
# ==========================================
# Инициализация SVD с заданными гиперпараметрами
model = SVD(
    n_factors = 50,  # Количество скрытых факторов
    n_epochs = 20,  # Количество эпох обучения (итераций SGD)
    lr_all = 0.005,  # Скорость обучения (learning rate) для всех параметров
    reg_all = 0.02,  # Коэффициент регуляризации для всех параметров
    random_state = 42,  # Фиксация случайного состояния для воспроизводимости инициализации
)

print("Обучение модели SVD...")
model.fit(trainset)

# ==========================================
# 4. Прогнозирование и оценка качества (RMSE, MAE)
# ==========================================
print("Тестирование на тестовой выборке...")
predictions = model.test(testset)

# Вычисление метрик точности предсказания
rmse_val = accuracy.rmse(predictions)
mae_val = accuracy.mae(predictions)

print(f"\nРезультаты оценки качества:")
print(f"RMSE: {rmse_val:.4f}")
print(f"MAE:  {mae_val:.4f}")

# ==========================================
# 5. Прогноз для конкретного пользователя и фильма
# ==========================================
# Важно: во встроенных датасетах Surprise идентификаторы пользователей (userId)
# и фильмов (movieId) по умолчанию считываются и хранятся как СТРОКИ ('str').
target_user = "1"
target_movie = "50"  # Фильм "Звездные войны: Эпизод 4" в ml-100k

# Генерация точечного прогноза
pred = model.predict(uid=target_user, iid=target_movie)

print(f"\nПрогноз для конкретной пары:")
print(f"Пользователь ID: {target_user}")
print(f"Фильм ID:        {target_movie}")
print(f"Предсказанный рейтинг (est): {pred.est:.4f}")

Загрузка встроенного датасета MovieLens-100k...
Dataset ml-100k could not be found. Do you want to download it? [Y/n] Y
Trying to download dataset from https://files.grouplens.org/datasets/movielens/ml-100k.zip...
Done! Dataset ml-100k has been saved to /root/.surprise_data/ml-100k
Обучение модели SVD...
Тестирование на тестовой выборке...
RMSE: 0.9348
MAE:  0.7377

Результаты оценки качества:
RMSE: 0.9348
MAE:  0.7377

Прогноз для конкретной пары:
Пользователь ID: 1
Фильм ID:        50
Предсказанный рейтинг (est): 4.6877


In [ ]:
# ==========================================
# 5.2 ДОБАВЛЕНА РАСШИФРОВКА НАЗВАНИЯ ЕСЛИ ХОЧЕТСЯ
# Просто замени кусок в разделе 5 на то что ниже
# ==========================================
import io
import os
from surprise import get_dataset_dir

target_user = "1"
target_movie = "50"

def get_movie_title(movie_id):
    u_item_path = os.path.join(get_dataset_dir(), "ml-100k/ml-100k/u.item")
    try:
        with io.open(u_item_path, "r", encoding="ISO-8859-1") as f:
            for line in f:
                parts = line.split("|")
                if parts[0] == str(movie_id):
                    return parts[1]
    except FileNotFoundError:
        return "Название неизвестно (файл u.item не найден)"
    return "Фильм не найден"

# Получение предсказания от обученной модели SVD
pred = model.predict(uid=target_user, iid=target_movie)
movie_title = get_movie_title(target_movie)

print(f"\n--- Единичное предсказание ---")
print(f"Пользователь ID: {target_user}")
print(f"Фильм ID:        {target_movie} ({movie_title})")
print(f"Предсказанный рейтинг (est): {pred.est:.4f}")


--- Единичное предсказание ---
Пользователь ID: 1
Фильм ID:        50 (Star Wars (1977))
Предсказанный рейтинг (est): 4.6877


# 7

7. Загрузите файлы ratings.csv и movies.csv из датасета MovieLens (ml-latest-small). Обратите внимание на колонку timestamp (время в формате Unix) в файле ratings.csv. Отсортируйте все записи по возрастанию времени (от старых к новым). Возьмите первые 80% записей как обучающую выборку, последние 20% как тестовую. Выведите количество записей в обучающей и тестовой выборках.

In [ ]:
import pandas as pd; import urllib.request; import zipfile; import os; import time; from datetime import datetime, timezone, timedelta
_t = time.time()
#Вместо принта, выводит текущее время и сколько прошло с прошлого принта
def p(m):
    global _t; now = datetime.now(timezone(timedelta(hours=3))); el = time.time() - _t; _t = time.time()
    h, r = divmod(el, 3600); m_val, s = divmod(r, 60)
    print(f"[{now.strftime('%H:%M:%S')}] {m}, elapsed {int(h)} hours {int(m_val)} minutes {int(s)} seconds")

if not os.path.exists('ml-latest-small'):
    p("Downloading dataset")
    urllib.request.urlretrieve('https://files.grouplens.org/datasets/movielens/ml-latest-small.zip', 'ml-latest-small.zip')
    with zipfile.ZipFile('ml-latest-small.zip', 'r') as zf: zf.extractall('.')
    p("Dataset downloaded and extracted")

df_r = pd.read_csv('ml-latest-small/ratings.csv')
# Отсортируйте все записи по возрастанию времени (от старых к новым).
df_r = df_r.sort_values('timestamp').reset_index(drop=True)
#Возьмите первые 80% записей как обучающую выборку, последние 20% как тестовую.
split = int(len(df_r) * 0.8)
train, test = df_r.iloc[:split], df_r.iloc[split:]
#Выведите количество записей в обучающей и тестовой выборках.
p(f"Train size: {len(train)}, Test size: {len(test)}")

[13:49:42] Downloading dataset, elapsed 0 hours 0 minutes 0 seconds
[13:49:43] Dataset downloaded and extracted, elapsed 0 hours 0 minutes 0 seconds
[13:49:43] Train size: 80668, Test size: 20168, elapsed 0 hours 0 minutes 0 seconds


# 8

8. Загрузите файлы ratings.csv и movies.csv из датасета MovieLens (ml-latest-small). Выберите одного пользователя (например, userId = 1). Отфильтруйте все его оценки и отсортируйте их по возрастанию timestamp. Извлеките список movieId в порядке просмотра. Выведите первые 10 фильмов в этой последовательности (их ID). Дополнительно, объедините с таблицей movies.csv, чтобы вывести названия этих 10 фильмов.

In [ ]:
import pandas as pd; import urllib.request; import zipfile; import os; import time; from datetime import datetime, timezone, timedelta
_t = time.time()
#Вместо принта, выводит текущее время и сколько прошло с прошлого принта
def p(m):
    global _t; now = datetime.now(timezone(timedelta(hours=3))); el = time.time() - _t; _t = time.time()
    h, r = divmod(el, 3600); m_val, s = divmod(r, 60)
    print(f"[{now.strftime('%H:%M:%S')}] {m}, elapsed {int(h)} hours {int(m_val)} minutes {int(s)} seconds")

if not os.path.exists('ml-latest-small'):
    p("Downloading dataset")
    urllib.request.urlretrieve('https://files.grouplens.org/datasets/movielens/ml-latest-small.zip', 'ml-latest-small.zip')
    with zipfile.ZipFile('ml-latest-small.zip', 'r') as zf: zf.extractall('.')
    p("Dataset downloaded and extracted")
# Загрузите файлы ratings.csv и movies.csv из датасета MovieLens (ml-latest-small).
df_r = pd.read_csv('ml-latest-small/ratings.csv')
df_m = pd.read_csv('ml-latest-small/movies.csv')

# Выберите одного пользователя (например, userId = 1).
# Отфильтруйте все его оценки и отсортируйте их по возрастанию timestamp.
u1 = df_r[df_r['userId'] == 1].sort_values('timestamp')

# Извлеките список movieId в порядке просмотра.
# Выведите первые 10 фильмов в этой последовательности (их ID).
top10_ids = u1['movieId'].head(10).tolist()
p(f"First 10 movie IDs for userId 1: {top10_ids}")

# Дополнительно, объедините с таблицей movies.csv, чтобы вывести названия этих 10 фильмов
top10_movies = df_m[df_m['movieId'].isin(top10_ids)].set_index('movieId').loc[top10_ids].reset_index()[['movieId', 'title']]
p(f"First 10 movies for userId 1:\n{top10_movies.to_string(index=False)}")

[13:49:43] First 10 movie IDs for userId 1: [804, 1210, 2018, 2826, 2628, 3578, 3617, 3744, 441, 101], elapsed 0 hours 0 minutes 0 seconds
[13:49:43] First 10 movies for userId 1:
 movieId                                             title
     804                              She's the One (1996)
    1210 Star Wars: Episode VI - Return of the Jedi (1983)
    2018                                      Bambi (1942)
    2826                          13th Warrior, The (1999)
    2628  Star Wars: Episode I - The Phantom Menace (1999)
    3578                                  Gladiator (2000)
    3617                                  Road Trip (2000)
    3744                                      Shaft (2000)
     441                         Dazed and Confused (1993)
     101                              Bottle Rocket (1996), elapsed 0 hours 0 minutes 0 seconds


# 11 🎏

11. Загрузите файлы ratings.csv и movies.csv из датасета MovieLens (ml-latest-small). Извлеките колонку title (названия фильмов) из файла movies.csv. Используйте TfidfVectorizer из sklearn.feature_extraction.text, чтобы преобразовать названия в TF-IDF признаки. Удалите стандартные английские стоп-слова (stop_words='english'). Выведите размер полученной матрицы (количество фильмов × размер словаря). Выведите 10 самых важных слов (с наибольшим TF-IDF) для первого фильма.


In [ ]:
import pandas as pd
import requests
import zipfile
import io
import os

# Ссылка на датасет MovieLens Small
url = 'https://files.grouplens.org/datasets/movielens/ml-latest-small.zip'

# Скачивание и распаковка, если папки еще нет
if not os.path.exists('ml-latest-small'):
    print("Скачиваем датасет...")
    r = requests.get(url)
    z = zipfile.ZipFile(io.BytesIO(r.content))
    z.extractall()
    print("Датасет успешно загружен и распакован!")
else:
    print("Датасет уже существует локально.")

Скачиваем датасет...
Датасет успешно загружен и распакован!


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# 1. Загрузка данных
# По условию просят загрузить оба файла, хотя для этой задачи ratings не нужен
movies = pd.read_csv('ml-latest-small/movies.csv')
ratings = pd.read_csv('ml-latest-small/ratings.csv')

# 2. Извлечение колонки title
titles = movies['title']

# 3-4. Преобразование в TF-IDF с удалением английских стоп-слов
vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix = vectorizer.fit_transform(titles)

# 5. Вывод размера полученной матрицы
print(f"Размер TF-IDF матрицы (фильмы x словарь): {tfidf_matrix.shape}")

# 6. Вывод 10 самых важных слов для первого фильма
first_movie_title = titles.iloc[0]
print(f"\nАнализируем первый фильм: '{first_movie_title}'")

# Получаем вектор первого фильма и названия всех слов в словаре
first_movie_vector = tfidf_matrix[0].toarray().flatten()
feature_names = vectorizer.get_feature_names_out()

# Создаем Series для удобной сортировки (Слово -> TF-IDF score)
word_scores = pd.Series(first_movie_vector, index=feature_names)

# Оставляем только те слова, вес которых больше 0 (которые реально есть в названии)
word_scores = word_scores[word_scores > 0]

# Сортируем по убыванию важности и берем топ-10
top_words = word_scores.sort_values(ascending=False).head(10)

print("Топ важных слов (с наибольшим TF-IDF):")
for word, score in top_words.items():
    print(f"- {word}: {score:.4f}")

Размер TF-IDF матрицы (фильмы x словарь): (9742, 9053)

Анализируем первый фильм: 'Toy Story (1995)'
Топ важных слов (с наибольшим TF-IDF):
- toy: 0.7496
- story: 0.5172
- 1995: 0.4130


# 12 🎏

12. Загрузите файлы ratings.csv и movies.csv из датасета MovieLens (ml-latest-small). Создайте гибридную систему для одного пользователя (например, userId = 1). Найдите топ-10 популярных фильмов (по числу оценок в ratings.csv). Найдите средний вектор жанров всех фильмов, которые этот пользователь оценил (из ratings.csv, используйте one-hot жанры). Затем найдите топ-10 фильмов (из ещё не оценённых пользователем), наиболее близких к этому среднему вектору по косинусному сходству.
Объедините два списка, убрав дубликаты (если фильм есть в обоих, оставьте одну запись). Выведите итоговые топ-10 рекомендаций в виде списка названий фильмов.


In [ ]:
import pandas as pd
import requests
import zipfile
import io
import os

# Ссылка на датасет MovieLens Small
url = 'https://files.grouplens.org/datasets/movielens/ml-latest-small.zip'

# Скачивание и распаковка, если папки еще нет
if not os.path.exists('ml-latest-small'):
    print("Скачиваем датасет...")
    r = requests.get(url)
    z = zipfile.ZipFile(io.BytesIO(r.content))
    z.extractall()
    print("Датасет успешно загружен и распакован!")
else:
    print("Датасет уже существует локально.")

# 15 😇

Загрузите файлы ratings.csv и movies.csv из датасета MovieLens (ml-latest-small). Постройте неявную матрицу пользователь–фильм (рейтинг ≥ 4 → 1). Используя scipy.sparse.csr_matrix, преобразуйте эту матрицу в разреженный формат CSR. Выведите: размер матрицы (пользователи × фильмы), количество ненулевых элементов, плотность матрицы (ненулевые / общее количество элементов).

In [ ]:
import pandas as pd
from scipy.sparse import csr_matrix
import requests
import zipfile
import io

url = 'https://files.grouplens.org/datasets/movielens/ml-latest-small.zip'
r = requests.get(url)
z = zipfile.ZipFile(io.BytesIO(r.content))

ratings = pd.read_csv(z.open('ml-latest-small/ratings.csv'))
movies = pd.read_csv(z.open('ml-latest-small/movies.csv'))

# Создание неявной матрицы: рейтинг >= 4 -> 1
# иначе удаляем (ставить 0 нет смысла, т.к. sparse всё равно сохранит только ненулевые значения)
implicit_ratings = ratings[ratings['rating'] >= 4].copy()
implicit_ratings['rating'] = 1

# Формирование сводной таблицы (матрицы)
# используем категориальные типы для корректного маппинга индексов в матрице
user_u = list(implicit_ratings.userId.unique())
movie_u = list(implicit_ratings.movieId.unique())

# Сопоставление ID с индексами
row = implicit_ratings.userId.astype('category').cat.codes
col = implicit_ratings.movieId.astype('category').cat.codes
data = implicit_ratings.rating.values

# Создание CSR матрицы
sparse_matrix = csr_matrix((data, (row, col)), shape=(len(user_u), len(movie_u)))

# Расчет характеристик
n_users, n_items = sparse_matrix.shape
n_nonzero = sparse_matrix.nnz
density = n_nonzero / (n_users * n_items)
print(f"Размер матрицы (пользователи x фильмы): {n_users} x {n_items}")
print(f"Количество ненулевых элементов: {n_nonzero}")
print(f"Плотность матрицы: {density:.6f}")

Размер матрицы (пользователи x фильмы): 609 x 6298
Количество ненулевых элементов: 48580
Плотность матрицы: 0.012666


# 16 😇

Загрузите файлы ratings.csv и movies.csv из датасета MovieLens (ml-latest-small). Разделите ratings.csv на обучающую (80%) и тестовую (20%) выборки случайным образом. На обучающей выборке определите топ-10 популярных фильмов (по числу оценок) — это ваши рекомендации. Выберите одного пользователя из тестовой выборки. Найдите все фильмы, которые он оценил в тесте. Найдите позицию первого фильма из этого списка в ваших рекомендациях (позиция считается с 1). Если такой фильм найден, MRR = 1 / позиция, иначе MRR = 0. Выведите значение MRR для этого пользователя.

In [ ]:
from sklearn.model_selection import train_test_split
import random
import pandas as pd
import requests
import zipfile
import io

url = 'https://files.grouplens.org/datasets/movielens/ml-latest-small.zip'
r = requests.get(url)
z = zipfile.ZipFile(io.BytesIO(r.content))

ratings = pd.read_csv(z.open('ml-latest-small/ratings.csv'))
movies = pd.read_csv(z.open('ml-latest-small/movies.csv'))

train_df, test_df = train_test_split(ratings, test_size=0.2, random_state=42)

# Топ-10 популярных фильмов в обучающей выборке (по количеству оценок)
top_10_movies = train_df['movieId'].value_counts().head(10).index.tolist()

# Выбор случайного пользователя из тестовой выборки
test_users = test_df['userId'].unique()
target_user = random.choice(test_users)

# Фильмы, которые этот пользователь оценил в тесте
user_test_movies = set(test_df[test_df['userId'] == target_user]['movieId'])

# Поиск позиции первого фильма из списка рекомендаций в тестовых оценках пользователя
mrr = 0
for i, movie_id in enumerate(top_10_movies, 1):
    if movie_id in user_test_movies:
        mrr = 1 / i
        break

print(f"ID пользователя: {target_user}")
print(f"Топ-10 рекомендаций (movieId): {top_10_movies}")
print(f"Фильмы пользователя в тесте: {list(user_test_movies)}")
print(f"MRR для пользователя: {mrr:.4f}")

ID пользователя: 276
Топ-10 рекомендаций (movieId): [356, 318, 296, 593, 2571, 260, 110, 480, 589, 1]
Фильмы пользователя в тесте: [480, 1, 2, 837, 135, 631, 780, 333, 783, 1367, 765, 62]
MRR для пользователя: 0.1250


MRR (Mean Reciprocal Rank). В данной задаче считаем RR для одного пользователя (если усреднить по всем, будет Mean).

Логика: рекомендации - это список. Пользователь обычно смотрит первые 3-5 предложенных вариантов, но до 50-й страницы рекомендаций мало кто доходит.
Нам важно, как высоко мы угадали:
- Если мы угадали фильм на 1-м месте, MRR = 1/1=1.0 (идеально)
- Если мы угадали на 10-м месте, MRR = 1/10=0.1 (плохо, пользователь мог не долистать)
- Если вообще не угадали в топ-10, MRR = 0 (очень плохо)

# 17

Загрузите файлы ratings.csv и movies.csv из датасета MovieLens (ml-latest-small). Для каждого фильма найдите множество пользователей, которые его оценили. Теперь для всех пар фильмов (i, j) вычислите количество общих пользователей (размер пересечения множеств). Постройте граф фильмов по совместным просмотрам: узлы — movieId. Добавьте ребро между фильмами i и j, если количество общих пользователей ≥ 5. Вес ребра — это количество общих пользователей. Выведите количество узлов (фильмов), количество рёбер и среднюю степень узла (среднее количество соседей у фильма).

In [ ]:
import pandas as pd
import networkx as nx
from itertools import combinations
from collections import Counter

# 1. Загрузка данных
# Подразумевается, что файлы 'ratings.csv' и 'movies.csv' находятся в одной папке со скриптом.
# Если файлы еще не скачаны, раскомментируйте блок ниже для их автоматической загрузки:

import urllib.request
from zipfile import ZipFile
print("Скачивание датасета ml-latest-small...")
url = "http://files.grouplens.org/datasets/movielens/ml-latest-small.zip"
urllib.request.urlretrieve(url, "ml-latest-small.zip")
with ZipFile("ml-latest-small.zip", "r") as zip_ref:
    zip_ref.extractall()


# Пути к файлам (замените на свои, если требуется)
movies_path = 'ml-latest-small/movies.csv'
ratings_path = 'ml-latest-small/ratings.csv'

movies_df = pd.read_csv(movies_path)
ratings_df = pd.read_csv(ratings_path)

# 2. Вычисление общих пользователей для пар фильмов
# Группируем фильмы по пользователям
user_movies = ratings_df.groupby('userId')['movieId'].apply(list)

# Считаем частоту встречаемости каждой пары фильмов
pair_counts = Counter()
for movies_list in user_movies:
    # Сортируем список для избежания дублирования пар вида (A, B) и (B, A)
    movies_sorted = sorted(movies_list)
    for m1, m2 in combinations(movies_sorted, 2):
        pair_counts[(m1, m2)] += 1

# Отбираем только те пары, у которых пересечение пользователей >= 5
edges = [(m1, m2, count) for (m1, m2), count in pair_counts.items() if count >= 5]

# 3. Построение графа
# При подсчете количества узлов метрики могут различаться в зависимости от того,
# какие именно узлы вы хотите включить в граф. Ниже приведены 3 возможных сценария:

# Сценарий А: Узлами являются ВСЕ фильмы из movies.csv (включая неимеющие ребер / изолированные)
G_all = nx.Graph()
G_all.add_nodes_from(movies_df['movieId'])
G_all.add_weighted_edges_from(edges)

# Сценарий Б: Узлами являются только те фильмы, которые были оценены хотя бы раз в ratings.csv
G_rated = nx.Graph()
G_rated.add_nodes_from(ratings_df['movieId'].unique())
G_rated.add_weighted_edges_from(edges)

# Сценарий В: Узлами являются только те фильмы, у которых есть ребра (степень >= 1)
# Узлы создаются автоматически на основе переданных ребер
G_connected = nx.Graph()
G_connected.add_weighted_edges_from(edges)


# Функция для расчета и вывода метрик графа
def print_graph_metrics(G, scenario_name):
    num_nodes = G.number_of_nodes()
    num_edges = G.number_of_edges()

    # Средняя степень узла: сумма степеней всех узлов, деленная на общее число узлов
    degrees = [deg for node, deg in G.degree()]
    avg_degree = sum(degrees) / num_nodes if num_nodes > 0 else 0

    print(f"=== {scenario_name} ===")
    print(f"Количество узлов (фильмов): {num_nodes}")
    print(f"Количество рёбер: {num_edges}")
    print(f"Средняя степень узла: {avg_degree:.2f}\n")

# Вывод результатов для каждого из возможных вариантов трактовки узлов
print_graph_metrics(G_all, "Вариант А: Все фильмы из каталога (movies.csv)")
print_graph_metrics(G_rated, "Вариант Б: Только фильмы с оценками (ratings.csv)")
print_graph_metrics(G_connected, "Вариант В: Только фильмы со связями (степень >= 1)")

Скачивание датасета ml-latest-small...
=== Вариант А: Все фильмы из каталога (movies.csv) ===
Количество узлов (фильмов): 9742
Количество рёбер: 1293963
Средняя степень узла: 265.65

=== Вариант Б: Только фильмы с оценками (ratings.csv) ===
Количество узлов (фильмов): 9724
Количество рёбер: 1293963
Средняя степень узла: 266.14

=== Вариант В: Только фильмы со связями (степень >= 1) ===
Количество узлов (фильмов): 3634
Количество рёбер: 1293963
Средняя степень узла: 712.14



# 18

Загрузите файлы ratings.csv и movies.csv из датасета MovieLens (ml-latest-small). Создайте неявную матрицу взаимодействий (рейтинг ≥ 4 → 1). Создайте признаки фильмов с применением one-hot encoding. Используйте класс Dataset из библиотеки Lightfm, чтобы: добавить всех пользователей и все фильмы, добавить признаки фильмов (жанры), построить матрицу взаимодействий. Создайте модель LightFM(loss='warp'). Обучите на 10 эпохах. Для пользователя userId = 1 получите топ-10 рекомендаций. Выведите названия этих фильмов.

In [ ]:
!pip install git+https://github.com/daviddavo/lightfm

In [ ]:
import pandas as pd
import numpy as np
from lightfm import LightFM
from lightfm.data import Dataset

# 1. Загрузка данных
# Предполагается, что файлы 'ratings.csv' и 'movies.csv' находятся в директории 'ml-latest-small/'
movies_path = 'ml-latest-small/movies.csv'
ratings_path = 'ml-latest-small/ratings.csv'

try:
    movies_df = pd.read_csv(movies_path)
    ratings_df = pd.read_csv(ratings_path)
except FileNotFoundError:
    # Пример автоматической загрузки, если датасет не скачан
    import urllib.request
    from zipfile import ZipFile
    print("Скачивание датасета ml-latest-small...")
    url = "http://files.grouplens.org/datasets/movielens/ml-latest-small.zip"
    urllib.request.urlretrieve(url, "ml-latest-small.zip")
    with ZipFile("ml-latest-small.zip", "r") as zip_ref:
        zip_ref.extractall()
    movies_df = pd.read_csv('ml-latest-small/movies.csv')
    ratings_df = pd.read_csv('ml-latest-small/ratings.csv')

# 2. Создание неявной матрицы взаимодействий (рейтинг >= 4 -> 1)
# Оставляем только положительные взаимодействия
ratings_implicit = ratings_df[ratings_df['rating'] >= 4.0].copy()

# 3. Создание признаков фильмов с применением One-Hot Encoding для жанров
# Разделяем жанры по разделителю '|' и преобразуем в OHE-представление
genres_ohe = movies_df['genres'].str.get_dummies(sep='|')
unique_genres = list(genres_ohe.columns)

# Формируем список кортежей (movieId, [список активных жанров]) для Dataset.build_item_features
item_features_list = []
for idx, row in movies_df.iterrows():
    movie_id = row['movieId']
    movie_genres_row = genres_ohe.iloc[idx]
    # Отбираем только те жанры, где значение признака равно 1
    active_genres = movie_genres_row[movie_genres_row == 1].index.tolist()
    item_features_list.append((movie_id, active_genres))

# 4. Использование класса Dataset из LightFM
dataset = Dataset()
# Фитируем датасет, передавая всех уникальных пользователей, фильмы и список жанров
dataset.fit(
    users=ratings_df['userId'].unique(),
    items=movies_df['movieId'].unique(),
    item_features=unique_genres
)

# Строим матрицу взаимодействий
interactions, weights = dataset.build_interactions(
    zip(ratings_implicit['userId'], ratings_implicit['movieId'])
)

# Строим матрицу признаков фильмов (размерность: n_items x (n_items + n_features))
item_features = dataset.build_item_features(item_features_list)

# 5. Обучение модели LightFM
# Используем функцию потерь WARP (Weighted Approximate-Rank Pairwise), эффективную для неявного фидбека
model = LightFM(loss='warp', random_state=42)
model.fit(
    interactions=interactions,
    item_features=item_features,
    epochs=10
)

# 6. Получение рекомендаций для userId = 1
# Получаем внутренние ID (индексы), с которыми работает LightFM
user_id_map, user_feature_map, item_id_map, item_feature_map = dataset.mapping()

internal_user_id = user_id_map[1]
all_external_movie_ids = list(item_id_map.keys())
all_internal_movie_ids = np.array(list(item_id_map.values()))

# Предсказываем скоры для всех фильмов из каталога
scores = model.predict(
    user_ids=internal_user_id,
    item_ids=all_internal_movie_ids,
    item_features=item_features
)

# Объединяем movieId, предсказанные скоры и названия фильмов
recommendations_df = pd.DataFrame({
    'movieId': all_external_movie_ids,
    'score': scores
})
recommendations_df = recommendations_df.merge(movies_df, on='movieId')

# Вариант А: Топ-10 без фильтрации уже оцененных фильмов
top_10_unfiltered = recommendations_df.sort_values(by='score', ascending=False).head(10)

# Вариант Б: Топ-10 с исключением фильмов, которые пользователь уже оценивал (более корректно для практики)
watched_movie_ids = ratings_df[ratings_df['userId'] == 1]['movieId'].unique()
top_10_filtered = recommendations_df[~recommendations_df['movieId'].isin(watched_movie_ids)].sort_values(by='score', ascending=False).head(10)

# Вывод результатов
print("=== Топ-10 рекомендаций для userId=1 (без фильтрации просмотренных) ===")
for i, row in enumerate(top_10_unfiltered.itertuples(), 1):
    print(f"{i}. {row.title} (Жанры: {row.genres})")

print("\n=== Топ-10 рекомендаций для userId=1 (исключая уже оцененные пользователем фильмы) ===")
for i, row in enumerate(top_10_filtered.itertuples(), 1):
    print(f"{i}. {row.title} (Жанры: {row.genres})")

=== Топ-10 рекомендаций для userId=1 (без фильтрации просмотренных) ===
1. Raiders of the Lost Ark (Indiana Jones and the Raiders of the Lost Ark) (1981) (Жанры: Action|Adventure)
2. Lord of the Rings: The Fellowship of the Ring, The (2001) (Жанры: Adventure|Fantasy)
3. Shawshank Redemption, The (1994) (Жанры: Crime|Drama)
4. Star Wars: Episode IV - A New Hope (1977) (Жанры: Action|Adventure|Sci-Fi)
5. Lord of the Rings: The Two Towers, The (2002) (Жанры: Adventure|Fantasy)
6. Matrix, The (1999) (Жанры: Action|Sci-Fi|Thriller)
7. Star Wars: Episode V - The Empire Strikes Back (1980) (Жанры: Action|Adventure|Sci-Fi)
8. Toy Story (1995) (Жанры: Adventure|Animation|Children|Comedy|Fantasy)
9. Star Wars: Episode VI - Return of the Jedi (1983) (Жанры: Action|Adventure|Sci-Fi)
10. Lord of the Rings: The Return of the King, The (2003) (Жанры: Action|Adventure|Drama|Fantasy)

=== Топ-10 рекомендаций для userId=1 (исключая уже оцененные пользователем фильмы) ===
1. Lord of the Rings: The Fellow

# 21.

Загрузите файл ratings.csv из датасета MovieLens (ml-latest-small). Постройте user-item матрицу, где строки – это идентификаторы пользователей, столбцы — идентификаторы фильмов, а значения – рейтинги, которые пользователь поставил фильму. Если пользователь не оценивал фильм, заполните пропуск нулём. Выберите двух любых пользователей (например, с userId = 1 и userId = 2). Извлеките их векторы оценок (строки матрицы). Вычислите косинусное сходство между этими двумя векторами. Обратите внимание: если оба вектора нулевые, сходство считается равным 0. Выведите полученное значение.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

ratings = pd.read_csv('ratings.csv')
user_item_matrix = ratings.pivot(index = 'userId', columns = 'movieId', values = 'rating').fillna(0)
user1_id = 1
user2_id = 2

# Извлекаем строки матрицы как numpy-массивы.
# Метод reshape(1, -1) нужен, так как sklearn ожидает двумерный массив.
user1_vector = user_item_matrix.loc[user1_id].values.reshape(1, -1)
user2_vector = user_item_matrix.loc[user2_id].values.reshape(1, -1)

if np.all(user1_vector == 0) or np.all(user2_vector == 0):
    similarity = 0.0
else:
    # cosine_similarity возвращает матрицу, нам нужен элемент [0][0]
    similarity = cosine_similarity(user1_vector, user2_vector)[0][0]

print(f"Косинусное сходство между пользователями: {similarity}")

# 22.

Загрузите файл movies.csv из датасета MovieLens (ml-latest-small). Поле genres содержит жанры, разделённые символом |. Преобразуйте жанры в бинарную матрицу (с применением one-hot кодирования): строки – фильмы, столбцы – уникальные жанры, значение 1, если фильм относится к жанру, иначе 0. Вычислите косинусное сходство между всеми фильмами на основе их жанров. Для фильма с movieId = 1 (Toy Story) найдите топ-5 самых похожих фильмов (не включая сам фильм). Выведите их названия и значения косинусного сходства.


In [ ]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

movies = pd.read_csv('movies.csv')
genres_encoded = movies['genres'].str.get_dummies('|')
cosine_sim_matrix = cosine_similarity(genres_encoded)

# Оборачиваем матрицу в DataFrame для удобного поиска по movieId
cosine_sim_df = pd.DataFrame(cosine_sim_matrix,
                             index = movies['movieId'],
                             columns = movies['movieId'])
target_movie_id = 1
similar_scores = cosine_sim_df[target_movie_id].drop(labels = [target_movie_id]) # Извлекаем строку сходств для Toy Story и удаляем сходство с самим собой (индекс 1)
top_5_similar = similar_scores.sort_values(ascending = False).head(5)

print(f"Топ-5 фильмов, похожих на фильм Toy Story:\n")
for movie_id, similarity in top_5_similar.items():
    movie_info = movies[movies['movieId'] == movie_id].iloc[0]
    print(f"- {movie_info['title']}")
    print(f"  Косинусное сходство: {similarity}\n")

# 23.

Загрузите данные ratings.csv датасета MovieLens (ml-latest-small) с помощью библиотеки Surprise через Reader и Dataset.load_from_df(). Раздели данные на обучающую (80%) и тестовую (20%) выборки. Создай модель KNNBasic с параметром user_based=True (чтобы искать похожих пользователей). Обучите модель на обучающей выборке. Сделайте прогноз для одной конкретной пары (например, userId = 1, movieId = 50). Выведите предсказанный рейтинг.

In [ ]:
!pip install surprise

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 58.4 MB/s eta 0:00:00


In [ ]:
import os
import urllib.request
import zipfile
import pandas as pd
from surprise import Dataset, Reader, KNNBasic
from surprise.model_selection import train_test_split

url = "https://files.grouplens.org/datasets/movielens/ml-latest-small.zip"
zip_path = "ml-latest-small.zip"
csv_path = "ml-latest-small/ratings.csv"

urllib.request.urlretrieve(url, zip_path)
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(".")

df = pd.read_csv(csv_path)

df_surprise = df[['userId', 'movieId', 'rating']]

reader = Reader(rating_scale=(0.5, 5.0))
data = Dataset.load_from_df(df_surprise, reader)

trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

sim_options = {
    'name': 'msd',
    'user_based': True
}
model = KNNBasic(sim_options=sim_options)

model.fit(trainset)

user_id = 1
movie_id = 50

prediction = model.predict(uid=user_id, iid=movie_id)

print("\nРезультаты прогнозирования:")
print(f"ID Пользователя: {user_id}")
print(f"ID Фильма: {movie_id}")
print(f"Предсказанный рейтинг: {prediction.est:.2f}")

Computing the msd similarity matrix...
Done computing similarity matrix.

Результаты прогнозирования:
ID Пользователя: 1
ID Фильма: 50
Предсказанный рейтинг: 4.62
